# Extend LUH: Data Exploration & Validation

Explore the input data and verify the extension strategy for **REMIND-MAgPIE 3.5-4.11, SSP1 - Very Low Emissions**.

1. AFOLU rampdown trajectory (CSV) and gridded rate-of-change
2. BECCS scaling curve and biofuel crop fractions
3. Prototype extension and sanity checks

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'extend-luh'))
# Also handle running from notebooks/ dir
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from src import config as cfg
from src.data import (
    load_csv, filter_scenario, get_variable,
    beccs_scaling_factors, afolu_ramp,
    load_states, load_biof, state_2100, rates_2100,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Load CSV and inspect AFOLU + BECCS trajectories

In [ ]:
df = load_csv()
sc = filter_scenario(df)

afolu = get_variable(sc, 'Emissions|CO2|AFOLU')
beccs = beccs_scaling_factors(sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
afolu.loc[2080:2200].plot(ax=ax, marker='.', ms=2)
ax.axhline(0, color='k', ls='--', lw=0.5)
ax.axvline(cfg.YR_AFOLU_ZERO, color='r', ls=':', label=f'AFOLU=0 @ {cfg.YR_AFOLU_ZERO}')
ax.set_ylabel('Mt CO2/yr'); ax.set_title('AFOLU CO2 emissions'); ax.legend()

ax = axes[1]
beccs.loc[2100:2500].plot(ax=ax, marker='.', ms=2)
ax.axhline(1, color='k', ls='--', lw=0.5)
ax.set_ylabel('Factor (rel. to 2100)'); ax.set_title('BECCS scaling factor')

plt.tight_layout(); plt.show()

## 2. Gridded state rates at 2100

In [ ]:
ds_states = load_states()
vals = state_2100(ds_states)
rates = rates_2100(ds_states)

# Global budget table
print(f'{"Variable":10s} {"2100 sum":>12s} {"Rate/yr":>12s} {"Δ by 2149":>12s} {"2149 est":>12s}')
print('-' * 60)
for v in cfg.STATE_VARS:
    s = np.nansum(vals[v])
    r = np.nansum(rates[v])
    delta = r * 49 / 2  # integral of linear ramp
    print(f'{v:10s} {s:12.1f} {r:12.2f} {delta:12.1f} {s+delta:12.1f}')

## 3. Rate ramp multiplier and AFOLU curve overlay

In [ ]:
yrs = np.arange(2100, 2200)
ramp = afolu_ramp(yrs)

# Normalise AFOLU to same scale for overlay
afolu_norm = afolu.loc[2100:2200] / afolu.loc[2100]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(yrs, ramp, 'b-', lw=2, label='Rate multiplier (linear ramp)')
ax.plot(afolu_norm.index, afolu_norm.values, 'r--', lw=1.5, label='AFOLU normalised')
ax.axvline(cfg.YR_AFOLU_ZERO, color='gray', ls=':')
ax.set_xlabel('Year'); ax.set_ylabel('Multiplier')
ax.set_title('Rate ramp vs AFOLU normalised trajectory')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Prototype extension (global sums)

In [ ]:
from src.extend import extend_states

ext_years = np.arange(2101, 2201)  # Just first 100 years for quick check
ext = extend_states(vals, rates, ext_years)

# Plot global sums
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Forest
ax = axes[0, 0]
for v, c in [('primf', 'darkgreen'), ('secdf', 'limegreen')]:
    # Historical (last 20 yr of input)
    hist = [np.nansum(ds_states[v].isel(time=t).values) for t in range(-20, 0)]
    hist_yrs = np.arange(2081, 2101)
    ext_sums = [np.nansum(ext[v][t]) for t in range(len(ext_years))]
    ax.plot(hist_yrs, hist, c=c, ls='--')
    ax.plot(ext_years, ext_sums, c=c, label=v)
ax.axvline(cfg.YR_AFOLU_ZERO, color='gray', ls=':')
ax.set_title('Forest'); ax.legend()

# Cropland
ax = axes[0, 1]
for v, c in [('c3ann', 'orange'), ('c3nfx', 'gold'), ('c4ann', 'sienna'), ('c4per', 'coral')]:
    hist = [np.nansum(ds_states[v].isel(time=t).values) for t in range(-20, 0)]
    hist_yrs = np.arange(2081, 2101)
    ext_sums = [np.nansum(ext[v][t]) for t in range(len(ext_years))]
    ax.plot(hist_yrs, hist, c=c, ls='--')
    ax.plot(ext_years, ext_sums, c=c, label=v)
ax.axvline(cfg.YR_AFOLU_ZERO, color='gray', ls=':')
ax.set_title('Crop types'); ax.legend()

# Total coverage check
ax = axes[1, 0]
total_ext = np.zeros(len(ext_years))
for v in cfg.STATE_VARS:
    total_ext += np.array([np.nansum(ext[v][t]) for t in range(len(ext_years))])
ax.plot(ext_years, total_ext)
ax.set_title('Total state sum (conservation check)')
ax.set_ylabel('Sum of fractions')

# secdf (absorbs excess)
ax = axes[1, 1]
hist_secdf = [np.nansum(ds_states['secdf'].isel(time=t).values) for t in range(-20, 0)]
ext_secdf = [np.nansum(ext['secdf'][t]) for t in range(len(ext_years))]
ax.plot(np.arange(2081, 2101), hist_secdf, 'g--')
ax.plot(ext_years, ext_secdf, 'g-', label='secdf')
ax.axvline(cfg.YR_AFOLU_ZERO, color='gray', ls=':')
ax.set_title('Secondary forest (absorbs excess)'); ax.legend()

plt.tight_layout(); plt.show()

## 5. BECCS biofuel scaling check

In [ ]:
from src.extend import extend_biofuel

ds_biof = load_biof()
crpbiof_2100 = ds_biof['crpbiof'].isel(time=-1).values

ext_years_full = np.arange(2101, 2501)
beccs_dict = beccs.to_dict()
crpbiof_ext = extend_biofuel(crpbiof_2100, beccs_dict, ext_years_full)

# Global sum of crpbiof over time
sums = [np.nansum(crpbiof_ext[t]) for t in range(len(ext_years_full))]
sum_2100 = np.nansum(crpbiof_2100)
achieved_factors = np.array(sums) / sum_2100

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ext_years_full, achieved_factors, 'b-', label='Achieved (with cap)')
target_factors = [beccs_dict.get(int(y), 1.0) for y in ext_years_full]
ax.plot(ext_years_full, target_factors, 'r--', label='Target (BECCS ratio)')
ax.set_xlabel('Year'); ax.set_ylabel('Factor rel. to 2100')
ax.set_title('Biofuel crop fraction: target vs achieved scaling')
ax.legend(); plt.tight_layout(); plt.show()

ds_biof.close()

In [ ]:
ds_states.close()